# ERY2-4 / CTLA-4 Docking Analysis
**Reference:** Ramanayake et al. *ACS Chem. Biol.* 2020, 15, 360–368  
**PDB:** 1I8L (Human B7-1/CTLA-4 complex)  
**Target:** CTLA-4 | **Peptide:** ERY2-4 (HLH, 40 aa)

---
## Key Experimental Data (from paper)
| Parameter | Value |
|-----------|-------|
| KD (SPR, 25°C) | 196.8 ± 2.3 nM |
| IC50 (SPR) | 1.1 ± 0.03 μM |
| IC50 (cell-based) | 631.9 ± 167.9 nM |
| CD28 cross-reactivity | None |
| Critical mutation | L33W (Trp33) |

In [ ]:
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# Experimental constants
EXPERIMENTAL_KD  = 196.8e-9   # Molar
EXPERIMENTAL_IC50_SPR  = 1.1e-6
EXPERIMENTAL_IC50_CELL = 631.9e-9
RT = 0.592  # kcal/mol at 25°C

exp_dg = RT * math.log(EXPERIMENTAL_KD)
print(f'Experimental KD  = {EXPERIMENTAL_KD*1e9:.1f} nM')
print(f'Experimental ΔG  = {exp_dg:.2f} kcal/mol')
print(f'Experimental IC50 (SPR)  = {EXPERIMENTAL_IC50_SPR*1e6:.1f} μM')
print(f'Experimental IC50 (cell) = {EXPERIMENTAL_IC50_CELL*1e9:.1f} nM')

## 1. Binding Affinity Comparison Across ERY2 Series

In [ ]:
# KD values from paper (SPR, synthetic peptides)
peptides = {
    'Y-2':    {'kd_nm': 6400,  'color': '#95a5a6', 'ic50_um': 11.2},
    'ERY2-1': {'kd_nm': 277.7, 'color': '#3498db', 'ic50_um': 0.92},
    'ERY2-4': {'kd_nm': 196.8, 'color': '#e74c3c', 'ic50_um': 1.1},
    'ERY2-6': {'kd_nm': 571.7, 'color': '#2ecc71', 'ic50_um': 1.7},
    'B7-1':   {'kd_nm': 200,   'color': '#f39c12', 'ic50_um': None},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: KD values
names = list(peptides.keys())
kds   = [peptides[n]['kd_nm'] for n in names]
colors = [peptides[n]['color'] for n in names]

bars = axes[0].bar(names, kds, color=colors, edgecolor='black', linewidth=1.2)
axes[0].axhline(y=200, color='orange', linestyle='--', linewidth=2, 
                label='B7-1 natural affinity (~200 nM)')
axes[0].set_ylabel('KD (nM)', fontsize=13, fontweight='bold')
axes[0].set_title('Binding Affinity to CTLA-4\n(SPR, synthetic peptides)', 
                   fontsize=13, fontweight='bold')
axes[0].set_yscale('log')
axes[0].legend(fontsize=10)

# Annotate ERY2-4
for bar, name, kd in zip(bars, names, kds):
    if name == 'ERY2-4':
        axes[0].annotate(f'{kd:.1f} nM\n★ Best binder',
                        xy=(bar.get_x() + bar.get_width()/2, kd),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', fontsize=9, fontweight='bold',
                        color='red')

# Plot 2: IC50 values
ic50_names  = [n for n in names if peptides[n]['ic50_um'] is not None]
ic50_values = [peptides[n]['ic50_um'] for n in ic50_names]
ic50_colors = [peptides[n]['color'] for n in ic50_names]

axes[1].bar(ic50_names, ic50_values, color=ic50_colors, 
            edgecolor='black', linewidth=1.2)
axes[1].set_ylabel('IC₅₀ (μM)', fontsize=13, fontweight='bold')
axes[1].set_title('Inhibition of CTLA-4/B7-1 Interaction\n(SPR, IC₅₀)', 
                   fontsize=13, fontweight='bold')

plt.suptitle('ERY2-4 HLH Peptide Series vs CTLA-4', 
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('data/output/06_visualization/binding_affinity_comparison.png', 
            dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved!')

## 2. CTLA-4 Binding Interface — MYPPPY Motif Analysis

In [ ]:
# CTLA-4 interface residues from 1I8L
interface_residues = {
    38:  {'name': 'Ile38',  'type': 'hydrophobic',    'mypppy': False},
    48:  {'name': 'Tyr48',  'type': 'hydrophobic',    'mypppy': False},
    50:  {'name': 'Glu50',  'type': 'electrostatic',  'mypppy': False},
    93:  {'name': 'Lys93',  'type': 'electrostatic',  'mypppy': False},
    97:  {'name': 'Met97',  'type': 'hydrophobic',    'mypppy': True},
    98:  {'name': 'Tyr98',  'type': 'pi_stacking',    'mypppy': True},
    99:  {'name': 'Pro99',  'type': 'MYPPPY',         'mypppy': True},
    100: {'name': 'Pro100', 'type': 'MYPPPY',         'mypppy': True},
    101: {'name': 'Pro101', 'type': 'MYPPPY',         'mypppy': True},
    102: {'name': 'Tyr102', 'type': 'key_contact',    'mypppy': True},
    110: {'name': 'Asn110', 'type': 'H_bond',         'mypppy': False},
}

type_colors = {
    'hydrophobic':   '#e74c3c',
    'electrostatic': '#3498db',
    'pi_stacking':   '#9b59b6',
    'MYPPPY':        '#e67e22',
    'key_contact':   '#c0392b',
    'H_bond':        '#1abc9c',
}

fig, ax = plt.subplots(figsize=(14, 5))

res_nums = sorted(interface_residues.keys())
bar_colors = [type_colors[interface_residues[r]['type']] for r in res_nums]
mypppy_flag = [interface_residues[r]['mypppy'] for r in res_nums]
labels = [interface_residues[r]['name'] for r in res_nums]

bars = ax.bar(range(len(res_nums)), [1]*len(res_nums), 
              color=bar_colors, edgecolor='black', linewidth=1.5, width=0.7)

# MYPPPY box
mypppy_start = res_nums.index(97)
mypppy_end   = res_nums.index(102)
ax.add_patch(mpatches.FancyBboxPatch(
    (mypppy_start - 0.4, -0.05), mypppy_end - mypppy_start + 0.8, 1.15,
    boxstyle='round,pad=0.1', fill=False,
    edgecolor='red', linewidth=3, linestyle='--'
))
ax.text((mypppy_start + mypppy_end)/2, 1.1, 'MYPPPY Motif',
        ha='center', fontsize=11, fontweight='bold', color='red')

ax.set_xticks(range(len(res_nums)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=11)
ax.set_yticks([])
ax.set_title('CTLA-4 Binding Interface Residues (PDB: 1I8L)\n'
             'ERY2-4 competes with B7-1 for this surface',
             fontsize=13, fontweight='bold')

# Legend
legend_patches = [mpatches.Patch(color=c, label=t.replace('_', ' ').title())
                  for t, c in type_colors.items()]
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('data/output/06_visualization/ctla4_interface_residues.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Interface figure saved!')

## 3. ERY2-4 Sequence Analysis — Key Residues

In [ ]:
sequence = 'CAWGSAILEGELAWLEGGGGGCGSQLADLKRQLAWSKQAC'

# Residue properties
residue_props = {
    'C': 'special',       # Cys — disulfide
    'A': 'nonpolar',      # Ala
    'W': 'aromatic',      # Trp — KEY
    'G': 'nonpolar',      # Gly
    'S': 'polar',         # Ser
    'I': 'nonpolar',      # Ile
    'L': 'nonpolar',      # Leu
    'E': 'negative',      # Glu
    'D': 'negative',      # Asp
    'K': 'positive',      # Lys
    'R': 'positive',      # Arg
    'Q': 'polar',         # Gln
}

prop_colors = {
    'special':  '#f39c12',
    'aromatic': '#e74c3c',
    'nonpolar': '#95a5a6',
    'polar':    '#3498db',
    'positive': '#9b59b6',
    'negative': '#2ecc71',
}

key_residues = {
    1:  'Cys1\n(SS)',
    3:  'Trp3',
    14: 'Trp14',
    22: 'Cys22\n(SS)',
    33: 'Trp33★\n(L33W)',
    39: 'Trp39',
    40: 'Cys40\n(SS)',
}

fig, ax = plt.subplots(figsize=(16, 4))

for i, aa in enumerate(sequence):
    pos = i + 1
    prop = residue_props.get(aa, 'nonpolar')
    color = prop_colors[prop]

    rect = mpatches.FancyBboxPatch(
        (i - 0.4, 0), 0.8, 1,
        boxstyle='round,pad=0.05',
        facecolor=color, edgecolor='black',
        linewidth=2 if pos in key_residues else 0.5
    )
    ax.add_patch(rect)
    ax.text(i, 0.5, aa, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    ax.text(i, -0.15, str(pos), ha='center', va='top',
            fontsize=6, color='black')

    if pos in key_residues:
        ax.annotate(key_residues[pos],
                   xy=(i, 1), xytext=(i, 1.4),
                   ha='center', fontsize=7,
                   fontweight='bold' if '★' in key_residues[pos] else 'normal',
                   color='red' if '★' in key_residues[pos] else 'black',
                   arrowprops=dict(arrowstyle='->', color='gray', lw=1))

# Disulfide arc
arc = mpatches.Arc((19.5, 0), 39, 1.5, angle=0,
                   theta1=0, theta2=180,
                   color='orange', linewidth=2.5, linestyle='--')
ax.add_patch(arc)
ax.text(19.5, 0.8, 'Disulfide bond (Cys1–Cys40)',
        ha='center', fontsize=9, color='orange', fontweight='bold')

ax.set_xlim(-1, 40)
ax.set_ylim(-0.4, 2.2)
ax.set_title('ERY2-4 Sequence Map\n'
             'Red = Trp33★ (L33W primary anchor) | Orange = Cys (disulfide)',
             fontsize=13, fontweight='bold')
ax.set_yticks([])

# Legend
legend_patches = [mpatches.Patch(color=c, label=t.title())
                  for t, c in prop_colors.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=8,
         bbox_to_anchor=(1, 0.1))

plt.tight_layout()
plt.savefig('data/output/06_visualization/ery24_sequence_map.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Sequence map saved!')

## 4. Load and Inspect Docking Results

In [ ]:
# Load analysis results (after running step 5)
results_file = Path('data/output/05_analysis/analysis_results.json')

if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)
    
    print('=== DOCKING RESULTS ===')
    
    # Interface
    if 'interface' in results:
        iface = results['interface']
        print(f"\nCTLA-4 interface residues: {iface['receptor_contacts']}")
        print(f"ERY2-4 interface residues: {iface['ligand_contacts']}")
    
    # MYPPPY
    if 'mypppy' in results:
        m = results['mypppy']
        print(f"\nMYPPPY coverage: {m['mypppy_coverage']*100:.0f}%")
        print(f"MYPPPY residues contacted: {m['mypppy_contacted']}")
    
    # Trp33
    if 'trp33' in results:
        t = results['trp33']
        print(f"\nTrp33 contacts: {t['trp33_contacts_found']}")
        print(f"Trp33 CTLA-4 residues: {t['trp33_ctla4_residues']}")
        print(f"Hits hydrophobic pocket: {t['hits_hydrophobic_pocket']}")
    
    # B7-1 overlap
    if 'b71_overlap' in results:
        ov = results['b71_overlap']
        print(f"\nB7-1 overlap: {ov['overlap_percent']}%")
        print(f"Shared residues: {ov['overlap_residues']}")
    
    # PRODIGY
    if 'prodigy' in results and results['prodigy'].get('available'):
        p = results['prodigy']
        print(f"\nPRODIGY predicted KD: {p.get('predicted_KD_nM', 'N/A'):.1f} nM")
        print(f"Experimental KD:      {EXPERIMENTAL_KD*1e9:.1f} nM")
else:
    print('No docking results yet.')
    print('Run: python3 scripts/05_analyze_results.py')

## 5. Affinity Maturation Progress

In [ ]:
# Affinity maturation — Y-2 -> ERY2-4
maturation_steps = [
    {'name': 'YT1-S\n(parent scaffold)', 'kd_nm': None,   'stage': 'scaffold'},
    {'name': 'Y-2\n(initial hit)',        'kd_nm': 6400,   'stage': 'initial'},
    {'name': 'ERY2-1\n(matured)',         'kd_nm': 277.7,  'stage': 'matured'},
    {'name': 'ERY2-4\n(best)',            'kd_nm': 196.8,  'stage': 'best'},
    {'name': 'ERY2-6\n(matured)',         'kd_nm': 571.7,  'stage': 'matured'},
]

stage_colors = {
    'scaffold': '#bdc3c7',
    'initial':  '#3498db',
    'matured':  '#2ecc71',
    'best':     '#e74c3c',
}

fig, ax = plt.subplots(figsize=(12, 6))

valid = [(m['name'], m['kd_nm'], stage_colors[m['stage']])
         for m in maturation_steps if m['kd_nm'] is not None]

names_v  = [v[0] for v in valid]
kds_v    = [v[1] for v in valid]
colors_v = [v[2] for v in valid]

bars = ax.bar(names_v, kds_v, color=colors_v, edgecolor='black',
              linewidth=1.5, width=0.6)

# B7-1 line
ax.axhline(y=200, color='orange', linewidth=2.5, linestyle='--',
           label=f'B7-1 natural affinity (~200 nM)')

# Fold improvement annotation
y2_kd = 6400
for bar, (name, kd, _) in zip(bars, valid):
    fold = y2_kd / kd
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 100,
            f'{kd:.0f} nM\n({fold:.0f}x)',
            ha='center', fontsize=9, fontweight='bold')

ax.set_ylabel('KD (nM) — lower is better', fontsize=13, fontweight='bold')
ax.set_title('Affinity Maturation: Y-2 → ERY2-4\n'
             '75-fold improvement over initial hit',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

# Arrow showing improvement
ax.annotate('', xy=(1.5, 300), xytext=(0.5, 5500),
            arrowprops=dict(arrowstyle='->', color='red',
                           lw=2, connectionstyle='arc3,rad=0.3'))
ax.text(0.8, 3000, '75x\nimprovement', fontsize=12,
        color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('data/output/06_visualization/affinity_maturation.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Affinity maturation figure saved!')

## 6. Summary

In [ ]:
print('='*60)
print('ERY2-4 / CTLA-4 DOCKING — ANALYSIS SUMMARY')
print('='*60)
print()
print('Peptide: ERY2-4 (HLH scaffold, 40 aa)')
print('Target:  CTLA-4 (PDB: 1I8L, chain C)')
print()
print('KEY FINDINGS:')
print(f'  1. KD = {EXPERIMENTAL_KD*1e9:.1f} nM (comparable to B7-1 ~200 nM)')
print(f'  2. IC50 = 1.1 μM (SPR competitive assay)')
print(f'  3. Trp33 (L33W) = primary hydrophobic anchor')
print(f'  4. Binding site = MYPPPY face of CTLA-4 (97-102)')
print(f'  5. No CD28 cross-reactivity (CTLA-4 selective)')
print(f'  6. 2-fold enhanced MLR response vs control')
print()
print('GENERATED FIGURES:')
for f in sorted(Path('data/output/06_visualization').glob('*.png')):
    print(f'  {f.name}')
print()
print('PYMOL SCRIPTS:')
for f in sorted(Path('data/output/06_visualization').glob('*.pml')):
    print(f'  pymol {f}')